In [1]:
import os
import math
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow INFO, WARNING, and ERROR logs
os.environ['TF_CUDNN_LOG_LEVEL'] = '3'    # Suppress cuDNN-related logs
os.environ['CUDA_LOG_LEVEL'] = '0'        # Suppress CUDA compiler logs
# Suppress GPU timer warnings
os.environ['TF_ENABLE_GPU_TIMER'] = 'false'

import warnings
warnings.filterwarnings('ignore', message='W0000 ')
warnings.filterwarnings('ignore', message="'+ptx85' is not a recognized feature for this target (ignoring feature)")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt
import cv2
import time
from skimage.metrics import structural_similarity as ssim

# random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)

# GPU memory management for 16GB VRAM
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f"Memory growth enabled for GPU: {gpus[0]}")
    except RuntimeError as e:
        print(e)

# global image size
IMG_SIZE = (512, 512)
IMG_WIDTH, IMG_HEIGHT = IMG_SIZE[0], IMG_SIZE[1]


BATCH_SIZE = 8

# image load and display
def load_raw_lunar_images_for_display(directory, num_images=500, img_size=IMG_SIZE):
    images = []
    for i, filename in enumerate(os.listdir(directory)):
        if i >= num_images:
            break
        img_path = os.path.join(directory, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, img_size, interpolation=cv2.INTER_NEAREST)
            images.append(img)
    images = np.array(images)
    images = images[..., np.newaxis]
    return images

# image preprocessing
def load_and_preprocess_lunar_images(directory, num_images=500, img_size=IMG_SIZE, augment=False):
    images = []
    for i, filename in enumerate(os.listdir(directory)):
        if i >= num_images:
            break
        img_path = os.path.join(directory, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, img_size, interpolation=cv2.INTER_NEAREST)
            if augment:
                if np.random.random() > 0.5:
                    img = cv2.flip(img, 1)
                img = img * np.random.uniform(0.8, 1.2)
            img = img.astype('float32') / 255.0
            images.append(img)
    images = np.array(images)
    images = images[..., np.newaxis]
    return images

# implementation of noise  
def add_noise(images, noise_factor=0.1):
    noisy = images + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=images.shape)
    return np.clip(noisy, 0.0, 1.0)

# calculating smoothness
def calculate_smoothness(image, window_size=8):
    h, w = image.shape[:2]
    smoothness_map = np.zeros((h, w))
    for i in range(0, h - window_size + 1, window_size):
        for j in range(0, w - window_size + 1, window_size):
            window = image[i:i + window_size, j:j + window_size]
            variance = np.var(window)
            smoothness_map[i:i + window_size, j:j + window_size] = variance
    return smoothness_map

# finding proper LZs  
def find_smoothest_region(smoothness_map, landing_window_size=8, surrounding_window_size=32):
    """
    Finds the best landing zone by considering both the smoothness of the
    immediate landing area and a larger surrounding area.

    Args:
        smoothness_map (np.ndarray): The map of variances calculated from the image.
        landing_window_size (int): The size of the final landing zone window.
        surrounding_window_size (int): The size of the larger area to check for overall smoothness.

    Returns:
        tuple: A tuple containing the best position (row, col) and the minimum combined variance.
    """
    h, w = smoothness_map.shape
    min_combined_variance = np.inf
    best_pos = (0, 0)
    
    # offset calculation
    offset = (surrounding_window_size - landing_window_size) // 2
    
    
    for i in range(offset, h - surrounding_window_size + offset + 1, landing_window_size):
        for j in range(offset, w - surrounding_window_size + offset + 1, landing_window_size):    
            # LZ smoothness calculation
            landing_zone_variance = np.mean(smoothness_map[i:i + landing_window_size, j:j + landing_window_size])
            
            # surrounding area smoothness calculation
            start_i = i - offset
            end_i = i + landing_window_size + offset
            start_j = j - offset
            end_j = j + landing_window_size + offset
            
            # constrain indices within bounds
            surrounding_area_variance = np.mean(smoothness_map[start_i:end_i, start_j:end_j])
            
            # variance combination
            # LZ 70% and surrounding area 30% 
            combined_variance = (0.7 * landing_zone_variance) + (0.3 * surrounding_area_variance)
            
            if combined_variance < min_combined_variance:
                min_combined_variance = combined_variance
                best_pos = (i, j)
                
    print(f"find_smoothest_region: best_pos {best_pos}, min_combined_variance {min_combined_variance}")
    return best_pos, min_combined_variance


# VAE sampling layer
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        z_mean = tf.cast(z_mean, tf.float32)
        z_log_var = tf.cast(z_log_var, tf.float32)
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim), dtype=tf.float32)
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# hyperspherical VAE sampling layer
class HypersphereSampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_kappa = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim), dtype=tf.float32)
        z = z_mean + tf.exp(z_kappa) * epsilon
        z_normalized = tf.math.l2_normalize(z, axis=-1)
        return z_normalized

# hyperbolic decoding!
# projection of a point from the hypersphere to a Euclidean space.
# the scale_factor  an be learned by the model to control the projection.
class HypersphereProjection(layers.Layer):
    def __init__(self, output_dim, **kwargs):
        super(HypersphereProjection, self).__init__(**kwargs)
        self.output_dim = output_dim
        
    def build(self, input_shape):
        self.scale = self.add_weight(
            name='scale',
            shape=(1,),
            initializer='ones',
            trainable=True,
            dtype=tf.float32
        )
        self.dense = layers.Dense(self.output_dim, use_bias=True)
        super(HypersphereProjection, self).build(input_shape)

    def call(self, inputs):
        # projection of the normalized HS input to a higher dimensional space
        x = inputs * self.scale
        return self.dense(x)


# modified KL divergence for the hyperspherical VAE
def hypersphere_kl_divergence(z_mean, z_kappa):
    kappa = tf.math.softplus(z_kappa)
    kl_loss = tf.reduce_mean(kappa - tf.math.log(kappa + 1e-8))
    return kl_loss

# Model Building Functions 

def build_hypersphere_vae(latent_dim=64):
    tf.keras.backend.clear_session()
    encoder_inputs = keras.Input(shape=(IMG_WIDTH, IMG_HEIGHT, 1))

    x = layers.Conv2D(64, 3, activation='relu', strides=2, padding='same')(encoder_inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(256, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(256, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(256, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    flatten_size = 16 * 16 * 256
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    
    z_mean = layers.Dense(latent_dim, name='z_mean')(x)
    z_kappa = layers.Dense(1, name='z_kappa')(x)
    
    z = HypersphereSampling()([z_mean, z_kappa])
    
    skip_placeholder = layers.Lambda(lambda t: t, name='skip_placeholder')(z)

    encoder = keras.Model(encoder_inputs, [z_mean, z_kappa, z, skip_placeholder], name='hypersphere_encoder')
    
    # change the decoding process to use HypersphereProjection 
    decoder_inputs = keras.Input(shape=(latent_dim,))
    
    # make the new layer to project from the hypersphere to a flat space
    latent_reshape_channels = 256 # match the first Conv2DTranspose layer for transfer
    x = HypersphereProjection(output_dim=16 * 16 * latent_reshape_channels)(decoder_inputs)
    x = layers.Reshape((16, 16, latent_reshape_channels))(x)
    
    x = layers.Conv2DTranspose(256, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2DTranspose(256, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2DTranspose(128, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2DTranspose(1, 3, activation='sigmoid', strides=2, padding='same')(x)
    
    decoder = keras.Model(decoder_inputs, x, name='hypersphere_decoder')
    decoder.summary()
    return encoder, decoder


# Complex VAE build with transfer learning
def build_complex_vae(pretrained_encoder, latent_dim=64):
    tf.keras.backend.clear_session()
    for layer in pretrained_encoder.layers:
        layer.trainable = True

    encoder_inputs = keras.Input(shape=(IMG_WIDTH, IMG_HEIGHT, 1))
    
    x = layers.Conv2D(64, 3, activation='relu', strides=2, padding='same')(encoder_inputs)
    x = layers.BatchNormalization()(x)
    conv_skip_256 = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(conv_skip_256)
    conv_skip_128 = layers.Conv2D(256, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(conv_skip_128)
    conv_skip_64 = layers.Conv2D(512, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(conv_skip_64)
    conv_skip_32 = layers.Conv2D(512, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(conv_skip_32)
    conv_skip_16 = layers.Conv2D(512, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(conv_skip_16)
    
    x = layers.Flatten()(x)
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    z_mean = layers.Dense(latent_dim, name='z_mean')(x)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
    z = Sampling()([z_mean, z_log_var])
    
    new_encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z, 
                                               [conv_skip_256, conv_skip_128, conv_skip_64, conv_skip_32, conv_skip_16]],
                              name='new_encoder')

    print("Attempting to transfer weights to complex VAE from hypersphere VAE.")
    # transfer weights from the hypersphere encoder to the complex VAE encoder
    for i, (pre_layer, new_layer) in enumerate(zip(pretrained_encoder.layers, new_encoder.layers)):
        if isinstance(pre_layer, layers.Conv2D) and isinstance(new_layer, layers.Conv2D):
            pre_weights = pre_layer.get_weights()
            new_weights = new_layer.get_weights()
            if len(pre_weights) == len(new_weights) and all(pw.shape == nw.shape for pw, nw in zip(pre_weights, new_weights)):
                try:
                    new_layer.set_weights(pre_weights)
                    print(f"Transferred weights to {new_layer.name} from {pre_layer.name}")
                except ValueError:
                    print(f"Could not transfer weights for {new_layer.name} due to shape mismatch (ValueError)")
                    pass
            else:
                print(f"Skipped weight transfer for {new_layer.name} due to shape mismatch")
                pass

    latent_inputs = keras.Input(shape=(latent_dim,))
    skip_inputs_256 = keras.Input(shape=(256, 256, 128))
    skip_inputs_128 = keras.Input(shape=(128, 128, 256))
    skip_inputs_64 = keras.Input(shape=(64, 64, 512))
    skip_inputs_32 = keras.Input(shape=(32, 32, 512))
    skip_inputs_16 = keras.Input(shape=(16, 16, 512))

    latent_reshape_channels = 512
    x = layers.Dense(16 * 16 * latent_reshape_channels, activation='relu')(latent_inputs)
    x = layers.Reshape((16, 16, latent_reshape_channels))(x)
    
    x = layers.Concatenate()([x, skip_inputs_16])
    x = layers.Conv2D(512, 3, activation='relu', padding='same')(x)

    x = layers.Conv2DTranspose(512, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Concatenate()([x, skip_inputs_32])
    x = layers.Conv2D(512, 3, activation='relu', padding='same')(x)

    x = layers.Conv2DTranspose(256, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Concatenate()([x, skip_inputs_64])
    x = layers.Conv2D(256, 3, activation='relu', padding='same')(x)

    x = layers.Conv2DTranspose(128, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Concatenate()([x, skip_inputs_128])
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)

    x = layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Concatenate()([x, skip_inputs_256])
    x = layers.Conv2DTranspose(1, 3, activation='sigmoid', strides=2, padding='same')(x)
    
    decoder = keras.Model([latent_inputs, skip_inputs_256, skip_inputs_128,
                           skip_inputs_64, skip_inputs_32, skip_inputs_16], x, name='complex_decoder')
    decoder.summary()
    return new_encoder, decoder

# VAE class with perceptual loss and SSIM (modified for hyperspherical VAE)
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name='total_loss')
        self.reconstruction_loss_tracker = keras.metrics.Mean(name='reconstruction_loss')
        self.kl_loss_tracker = keras.metrics.Mean(name='kl_loss')
        
        vgg_input = keras.Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3))
        vgg = VGG16(include_top=False, weights='imagenet', input_tensor=vgg_input)
        self.perceptual_model = Model(vgg_input, vgg.layers[1].output)
        self.perceptual_model.trainable = False
        
        self.mse_weight = 1.0
        self.perceptual_weight = 0.1
        self.kl_weight = 0.1

    def perceptual_loss(self, y_true, y_pred):
        y_true_rgb = tf.image.grayscale_to_rgb(y_true)
        y_pred_rgb = tf.image.grayscale_to_rgb(y_pred)
        y_true_rgb = tf.keras.applications.vgg16.preprocess_input(y_true_rgb * 255.0)
        y_pred_rgb = tf.keras.applications.vgg16.preprocess_input(y_pred_rgb * 255.0)
        
        true_features = self.perceptual_model(y_true_rgb)
        pred_features = self.perceptual_model(y_pred_rgb)
        return tf.reduce_mean(tf.square(true_features - pred_features))

    def call(self, inputs, training=False):
        if len(self.decoder.inputs) == 1:
            if 'hypersphere' in self.encoder.name:
                z_mean, z_kappa, z, _ = self.encoder(inputs)
            else:
                z_mean, z_log_var, z, _ = self.encoder(inputs)
            reconstruction = self.decoder(z)
        else:
            z_mean, z_log_var, z, skips = self.encoder(inputs)
            reconstruction = self.decoder([z] + skips)
        return reconstruction

    def train_step(self, data):
        if isinstance(data, tuple):
            inputs, targets = data
        else:
            inputs, targets = data, data

        with tf.GradientTape() as tape:
            if 'hypersphere' in self.encoder.name:
                # modifief HS VAE Loss
                z_mean, z_kappa, z, _ = self.encoder(inputs)
                reconstruction = self.decoder(z)
                kl_loss = hypersphere_kl_divergence(z_mean, z_kappa)
            else:
                # standard VAE Loss 
                z_mean, z_log_var, z, skips = self.encoder(inputs)
                reconstruction = self.decoder([z] + skips)
                kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
                kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))

            reconstruction_loss = tf.reduce_mean(tf.reduce_sum(tf.keras.losses.MSE(targets, reconstruction), axis=(1, 2)))
            perceptual_loss = self.perceptual_loss(targets, reconstruction)
            
            total_loss = (self.mse_weight * reconstruction_loss +
                          self.perceptual_weight * perceptual_loss +
                          self.kl_weight * kl_loss)
                          
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            'loss': self.total_loss_tracker.result(),
            'reconstruction_loss': self.reconstruction_loss_tracker.result(),
            'kl_loss': self.kl_loss_tracker.result(),
        }

    def test_step(self, data):
        if isinstance(data, tuple):
            inputs, targets = data
        else:
            inputs, targets = data, data

        if 'hypersphere' in self.encoder.name:
            z_mean, z_kappa, z, _ = self.encoder(inputs)
            reconstruction = self.decoder(z)
            kl_loss = hypersphere_kl_divergence(z_mean, z_kappa)
        else:
            z_mean, z_log_var, z, skips = self.encoder(inputs)
            reconstruction = self.decoder([z] + skips)
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            
        reconstruction_loss = tf.reduce_mean(tf.reduce_sum(tf.keras.losses.MSE(targets, reconstruction), axis=(1, 2)))
        perceptual_loss = self.perceptual_loss(targets, reconstruction)
        total_loss = (self.mse_weight * reconstruction_loss +
                      self.perceptual_weight * perceptual_loss +
                      self.kl_weight * kl_loss)
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            'loss': self.total_loss_tracker.result(),
            'reconstruction_loss': self.reconstruction_loss_tracker.result(),
            'kl_loss': self.kl_loss_tracker.result(),
        }

# Plot total loss vs. epochs
def plot_loss_history(history_simple, history_complex):
    plt.figure(figsize=(10, 5))
    plt.plot(history_simple.history['loss'], label='Hypersphere VAE Train Loss', color='blue')
    plt.plot(history_simple.history['val_loss'], label='Hypersphere VAE Val Loss', linestyle='--', color='blue')
    plt.plot(history_complex.history['loss'], label='Complex VAE Train Loss', color='red')
    plt.plot(history_complex.history['val_loss'], label='Complex VAE Val Loss', linestyle='--', color='red')
    plt.xlabel('Epoch')
    plt.ylabel('Total Loss')
    plt.title(f'Total Loss vs. Epochs for Hypersphere and Complex VAEs ({IMG_WIDTH}x{IMG_HEIGHT})')
    plt.legend()
    plt.grid(True)
    plt.savefig(f'loss_vs_epochs_hypersphere_{IMG_WIDTH}x{IMG_HEIGHT}.png')
    plt.show()

# PSNR calculation 
def calculate_psnr(image1, image2):
    
    mse = np.mean((image1.astype(np.float32) - image2.astype(np.float32)) ** 2)
    if mse == 0:
        return 100 # Perfect match
    max_pixel = 255.0
    psnr = 20 * math.log10(max_pixel / math.sqrt(mse))
    return psnr

# Main Execution 

# data load
data_dir = 'project data'
lunar_images_raw = load_raw_lunar_images_for_display(data_dir, num_images=500)
lunar_images = load_and_preprocess_lunar_images(data_dir, num_images=500)
lunar_noisy_images = add_noise(lunar_images)
print("lunar_images mean:", np.mean(lunar_images), "min:", np.min(lunar_images), "max:", np.max(lunar_images))
print("lunar_noisy_images mean:", np.mean(lunar_noisy_images), "min:", np.min(lunar_noisy_images), "max:", np.max(lunar_noisy_images))
print(f"lunar_images_raw shape: {lunar_images_raw.shape}")

# sample for inspection
cv2.imwrite(f'sample_lunar_image_{IMG_WIDTH}x{IMG_HEIGHT}.png', lunar_images_raw[0, :, :, 0])

# HS VAE train
tf.keras.backend.clear_session()
latent_dim = 64
encoder_hypersphere, decoder_hypersphere = build_hypersphere_vae(latent_dim)
hypersphere_vae = VAE(encoder_hypersphere, decoder_hypersphere)
hypersphere_vae.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005))
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

print(f"\n--- Starting Hypersphere VAE training on {IMG_WIDTH}x{IMG_HEIGHT} images (Batch Size: {BATCH_SIZE}) ---")
start_time = time.time()
history_hypersphere = hypersphere_vae.fit(lunar_noisy_images, lunar_images, epochs=100, batch_size=BATCH_SIZE,
                                         validation_split=0.2, callbacks=[early_stopping, lr_scheduler], verbose=1)
end_time = time.time()
print(f"Hypersphere VAE training time: {end_time - start_time:.2f} seconds")

# Complex VAE train
tf.keras.backend.clear_session()
# pass the hypersphere_vae encoder for potential weight initialization.
encoder_complex, decoder_complex = build_complex_vae(encoder_hypersphere, latent_dim=64)
complex_vae = VAE(encoder_complex, decoder_complex)
complex_vae.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005))
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

print(f"\n--- Starting Complex VAE training on {IMG_WIDTH}x{IMG_HEIGHT} images (Batch Size: {BATCH_SIZE}) ---")
start_time = time.time()
history_complex = complex_vae.fit(lunar_noisy_images, lunar_images, epochs=100, batch_size=BATCH_SIZE,
                                 validation_split=0.2, callbacks=[early_stopping, lr_scheduler], verbose=1)
end_time = time.time()
print(f"Complex VAE training time: {end_time - start_time:.2f} seconds")

# Plot loss history
plot_loss_history(history_hypersphere, history_complex)

# image reconstruction using complex VAE
reconstructions_complex = []
for i in range(0, len(lunar_noisy_images), BATCH_SIZE):
    batch = lunar_noisy_images[i:i + BATCH_SIZE]
    _, _, z, skips = complex_vae.encoder(batch)
    reconstruction_inputs = [z] + skips
    reconstruction = complex_vae.decoder(reconstruction_inputs).numpy()
    rec = np.clip(reconstruction * 255.0, 0, 255).astype(np.uint8)
    reconstructions_complex.append(rec)
reconstructions_complex = np.concatenate(reconstructions_complex, axis=0)

# visualisation of original, noisy, reconstructed, smoothness map, and smoothest region
num_examples = 5
plt.figure(figsize=(25, 5 * num_examples)) 
ssim_scores_noisy = []
ssim_scores_reconstructed = []
psnr_scores_noisy = []
psnr_scores_reconstructed = []
for i in range(num_examples):
    noisy = lunar_noisy_images[i] * 255.0
    clean = lunar_images[i] * 255.0
    reconstructed = reconstructions_complex[i]
    
    # smoothness with a default 8x8 window for the LZ
    smoothness_map = calculate_smoothness(reconstructed[..., 0])
    
    # tracking the optimal region using both the 8x8 and 32x32 window sizes
    best_pos, min_variance = find_smoothest_region(smoothness_map, landing_window_size=8, surrounding_window_size=32)
    print(f"Image {i+1} shape: {reconstructed.shape}")

    overlay_image = np.repeat(reconstructed[..., 0][..., np.newaxis], 3, axis=-1)
    overlay_image = overlay_image.astype(np.uint8)

    h, w, _ = overlay_image.shape
    y, x = best_pos[0], best_pos[1]
    
    # LZ border (red)
    landing_window_size = 8
    border_thickness = 1
    red_color = [255, 0, 0]
    
    # top-bottom
    overlay_image[y : y + border_thickness, x : x + landing_window_size, :] = red_color
    overlay_image[y + landing_window_size - border_thickness : y + landing_window_size, x : x + landing_window_size, :] = red_color
    
    # left-right
    overlay_image[y : y + landing_window_size, x : x + border_thickness, :] = red_color
    overlay_image[y : y + landing_window_size, x + landing_window_size - border_thickness : x + landing_window_size, :] = red_color

    # surrounding area border (blue)
    surrounding_window_size = 32
    blue_color = [0, 0, 255]
    
    # centering the LZ border in the surrounding area border
    surrounding_y = max(0, y - (surrounding_window_size - landing_window_size) // 2)
    surrounding_x = max(0, x - (surrounding_window_size - landing_window_size) // 2)

    # bounding condition 
    surrounding_y = min(surrounding_y, h - surrounding_window_size)
    surrounding_x = min(surrounding_x, w - surrounding_window_size)
    
    # top bottom
    overlay_image[surrounding_y : surrounding_y + border_thickness, surrounding_x : surrounding_x + surrounding_window_size, :] = blue_color
    overlay_image[surrounding_y + surrounding_window_size - border_thickness : surrounding_y + surrounding_window_size, surrounding_x : surrounding_x + surrounding_window_size, :] = blue_color
    
    # left right
    overlay_image[surrounding_y : surrounding_y + surrounding_window_size, surrounding_x : surrounding_x + border_thickness, :] = blue_color
    overlay_image[surrounding_y : surrounding_y + surrounding_window_size, surrounding_x + surrounding_window_size - border_thickness : surrounding_x + surrounding_window_size, :] = blue_color

    ssim_noisy = ssim(clean[..., 0], noisy[..., 0], data_range=255.0, win_size=7)
    ssim_reconstructed = ssim(clean[..., 0], reconstructed[..., 0], data_range=255.0, win_size=7)
    ssim_scores_noisy.append(ssim_noisy)
    ssim_scores_reconstructed.append(ssim_reconstructed)
    
    psnr_noisy = calculate_psnr(clean[..., 0], noisy[..., 0])
    psnr_reconstructed = calculate_psnr(clean[..., 0], reconstructed[..., 0])
    psnr_scores_noisy.append(psnr_noisy)
    psnr_scores_reconstructed.append(psnr_reconstructed)

    # New subplot for the original image
    plt.subplot(num_examples, 5, i * 5 + 1)
    plt.imshow(clean[..., 0], cmap='gray', vmin=0, vmax=255)
    plt.title(f'Original image {IMG_WIDTH}x{IMG_HEIGHT}')
    plt.axis('off')

    plt.subplot(num_examples, 5, i * 5 + 2)
    plt.imshow(noisy[..., 0], cmap='gray', vmin=0, vmax=255)
    plt.title(f'Noisy {i+1}\nSSIM: {ssim_noisy:.4f}\nPSNR: {psnr_noisy:.2f}dB')
    plt.axis('off')

    plt.subplot(num_examples, 5, i * 5 + 3)
    plt.imshow(reconstructed[..., 0], cmap='gray', vmin=0, vmax=255)
    plt.title(f'Reconstructed {i+1}\nSSIM: {ssim_reconstructed:.4f}\nPSNR: {psnr_reconstructed:.2f}dB')
    plt.axis('off')

    plt.subplot(num_examples, 5, i * 5 + 4)
    plt.imshow(smoothness_map, cmap='viridis')
    plt.title(f'Smoothness Map {i+1}\n(Variance: {min_variance:.2f})')
    plt.axis('off')

    plt.subplot(num_examples, 5, i * 5 + 5)
    plt.imshow(overlay_image)
    plt.title(f'Smoothest Region {i+1}')
    plt.axis('off')

plt.tight_layout()
plt.savefig(f'reconstruction_comparison_hypersphere_{IMG_WIDTH}x{IMG_HEIGHT}.png')
plt.show()
print(f"Average SSIM (Noisy vs. Clean): {np.mean(ssim_scores_noisy):.4f}")
print(f"Average SSIM (Reconstructed vs. Clean): {np.mean(ssim_scores_reconstructed):.4f}")
print(f"Average PSNR (Noisy vs. Clean): {np.mean(psnr_scores_noisy):.2f} dB")
print(f"Average PSNR (Reconstructed vs. Clean): {np.mean(psnr_scores_reconstructed):.2f} dB")

ModuleNotFoundError: No module named 'tensorflow'